# PyField water walkthrough — `relax_method: relaxed_constrained` end-to-end

A real polyatomic ReaxFF refit driving every piece of PyField:

1. **`qm-relax`** finds equilibrium geometries (single H₂O ≈ 0.96 Å / 104.5°; cyclic (H₂O)₆ ring ≈ 2.7 Å O–O).
2. **`make-scan`** with three relaxed-constrained scans:
   - O–H bond stretch on a single H₂O (5 distances, 0.85–1.40 Å);
   - H–O–H angle bend (6 angles, 80°–130°);
   - **Ring-opening on a (H₂O)₆ hexamer** — pull water 0 out of an ice-like loop by stretching its O…O distance to a neighbour (5 distances, 2.6–4.5 Å). Captures cooperative-network energetics — an H-bond inside a ring is ~1.5× stronger than a bare dimer due to polarisation feedback.
3. **Inline animation** of the ring-opening scan via `pyfield.viz`.
4. **`qm-prep`** runs B3LYP/def2-SVP constrained relaxes — at every scan point, QM holds the reaction coordinate fixed and relaxes everything else.
5. **Optimiser refit** (whatever `optimizer.method:` says — `sa`, `ga`, `sa+ga`, or `cma`) on 11 trainable parameters (`tests/params_HO`) — H–O bond strength + curve shape, H–O–H angle, H–O off-diagonal vdW, and per-element QEq electronegativity / hardness.
6. **`cost_breakdown`** before/after so you can see exactly which residuals shrunk.

Starting force field: `tests/ffield.reax.HO` — the LAMMPS-bundled Chenoweth/van Duin/Goddard 2008 c/h/o combustion ReaxFF. Carbon parameters are dormant since our YAML only has H/O atoms.

This notebook is heavy (16 constrained DFT relaxes + many parallel LAMMPS evaluations); excluded from the default `pytest`. Run it manually:
```
pytest examples/water_walkthrough.ipynb --nbmake --nbmake-timeout=1800
```
Requires `pip install -e .[dev,cma]` and a working LAMMPS (`pip install 'lammps[mpi]'`).

## 1. Load `tests/water_train.yaml`

Two starting structures (single H₂O + cyclic (H₂O)₆ hexamer ring) flagged `qm_relax: true`, three relaxed-constrained scans, an optimiser block, and the H/O ffield + 11-parameter `params_HO` file.

In [ ]:
import os, shutil
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
os.chdir(ROOT)
from pathlib import Path
from pyfield.config.loader import load_yaml

cfg = load_yaml('tests/water_train.yaml')
print(f'qm:          {cfg.qm.code}/{cfg.qm.functional}/{cfg.qm.basis}')
print(f'structures:  {len(cfg.structures)}  (qm_relax flagged: {[n for n, s in cfg.structures.items() if s.qm_relax]})')
print(f'scans:       {len(cfg.scans)}')
for s in cfg.scans:
    n = len(s.values) if s.values else s.range[2]
    print(f'             {s.type:<19} prefix={s.name_prefix:<14} N={n} relax={s.relax_method}')
print(f'optimiser:   method={cfg.optimizer.method}, parallel={cfg.optimizer.parallel}, '
      f'processors={cfg.optimizer.processors}')

## 2. `qm-relax` — equilibrium geometries

PySCF + geomeTRIC find the H₂O minimum (~0.96 Å O–H, ~103° H–O–H) and the cyclic (H₂O)₆ ring (planar S6 form at this level of theory; O–O ≈ 2.66 Å with all six H-bonds at ~1.67 Å). Wall-clock with cold cache: ~30 s for H₂O + ~2 min for the hexamer.

In [ ]:
import importlib.util
assert importlib.util.find_spec('pyscf') and importlib.util.find_spec('geometric'), \
    'this notebook requires `pip install -e .[dev]` (pulls pyscf + geometric)'

from pyfield.qm.prep import cfg_to_yaml, relax_structures

relaxed_cfg, journal = relax_structures(cfg)
for action, hit, key in journal:
    tag = '[cache hit ]' if hit else '[running   ]'
    print(f'  {tag} {action}   ({key})')

relaxed_path = Path('tests/water_train.relaxed.yaml')
relaxed_path.write_text(cfg_to_yaml(relaxed_cfg))

import math, numpy as np
h2o = relaxed_cfg.structures['H2O_Opt']
o, h1, h2 = h2o.atoms
v1 = np.array([h1.x-o.x, h1.y-o.y, h1.z-o.z])
v2 = np.array([h2.x-o.x, h2.y-o.y, h2.z-o.z])
ang = math.degrees(math.acos(np.dot(v1,v2)/(np.linalg.norm(v1)*np.linalg.norm(v2))))
print(f'\nrelaxed H2O:    O-H = {np.linalg.norm(v1):.4f} A,  H-O-H = {ang:.2f} deg')

ring = relaxed_cfg.structures['H2O6_ring_Opt']
oo = []
for i in range(6):
    a = ring.atoms[3*i]
    b = ring.atoms[3*((i+1) % 6)]
    oo.append(math.sqrt((a.x-b.x)**2 + (a.y-b.y)**2 + (a.z-b.z)**2))
oo_fmt = ', '.join('{:.3f}'.format(d) for d in oo)
print(f'relaxed (H2O)6: O-O ring distances = [{oo_fmt}] A')
print(f'                mean O-O = {sum(oo)/len(oo):.3f} A')

## 3. `make-scan` — stamp out 16 perturbed structures

Each scan kind expands into N structures + N `minimize` simulations (carrying the matching `fix restrain` string for the FF side) + N `energy_combination` targets. The QM-side constraint is also stashed on each generated structure so `qm-prep` can drive a constrained relax.

In [ ]:
from pyfield.scans import expand_scans

scanned_cfg, summary = expand_scans(relaxed_cfg, xyz_dir=Path('tests/runs/water_xyz'))
for line in summary:
    print(' ', line)

scanned_path = Path('tests/water_train.scanned.yaml')
scanned_path.write_text(cfg_to_yaml(scanned_cfg))

print()
print(f'structures: {len(scanned_cfg.structures)}  '
      f'(2 references + 16 scan points)')
print(f'simulations: {len(scanned_cfg.simulations)}  '
      f'(2 reference single_points + 16 constrained minimisations)')
print(f'targets:     {len(scanned_cfg.targets)}  energy_combination, all from: dft')

# Show one representative restraint string + one constraint spec from the
# ring-opening scan.
ex_sim = scanned_cfg.simulations['H2O6_open_2_sp']
ex_struct = scanned_cfg.structures['H2O6_open_2']
print(f'\nexample sim H2O6_open_2_sp:')
print(f'  type:       {ex_sim.type}')
print(f'  restraints: {ex_sim.__pydantic_extra__["restraints"]}')
print(f'\nexample structure H2O6_open_2:')
print(f'  qm_relax:   {ex_struct.qm_relax}')
print(f'  constraint: {ex_struct.__pydantic_extra__["constraint"]}')

## 4. Animate the ring-opening scan inline

Use `pyfield.viz.animate_xyz_dir` to spot-check the perturbations before paying for QM. Water 0 (atoms 1–3) is dragged out from the ring along the O₀–O₁ axis; the rest of the hexamer rides with its own anchor (O₁) during the perturbation. QM then relaxes everything else at each fixed O…O distance.

In [ ]:
import matplotlib
matplotlib.use('Agg')
from pyfield.viz import animate_xyz_dir

animate_xyz_dir(
    'tests/runs/water_xyz',
    pattern='H2O6_open_*.xyz',
    interval_ms=500,
    title='(H₂O)₆ ring-opening — O₀…O₁ stretch',
)

## 5. `qm-prep` — constrained relax at every scan point

For each of the 16 scan points, PySCF + geomeTRIC runs a constrained geometry optimisation: the reaction coordinate (O–H distance, H–O–H angle, or O₀–O₁ ring distance) is held at the per-point value, while every other atom relaxes to minimise the QM energy. That's the natural starting point for fitting an FF — both QM and FF (via `fix restrain`) sit at the same constrained-relaxed geometry, just with their own minima.

Wall-clock note: 5 H₂O scan-point relaxes are fast (~30 s each); the 5 hexamer ring-opening points are heavier (~2–4 min each) because the relax has to reorganise the entire 18-atom ring at each fixed O…O distance. The QM cache makes re-runs free.

In [ ]:
from pyfield.qm.prep import populate_qm

populated, journal = populate_qm(scanned_cfg)
ran = sum(1 for _, hit, _ in journal if not hit)
cached = len(journal) - ran
print(f'qm-prep: {ran} ran, {cached} cached, {len(journal)} total')

populated_path = Path('tests/water_train.populated.yaml')
populated_path.write_text(cfg_to_yaml(populated))

print('\nDFT ΔE targets (relative to reference):')
for tgt in populated.targets:
    extras = tgt.__pydantic_extra__
    sim_id = next(iter(extras['terms']))   # the +1 sim
    print(f'  {sim_id:<22}  ΔE = {extras["target"]:>10.4f} kcal/mol')

## 6. Optimiser refit with before/after `cost_breakdown`

11 trainable parameters, 16 targets, parallel evaluation.
`run_optimizer` dispatches on `cfg.optimizer.method` so the same notebook works for `sa`, `ga`, `sa+ga`, or `cma` without code changes.
Before-and-after `cost_breakdown` shows the per-target FF residual: total cost is just the sum of those residuals.

In [ ]:
from pyfield.io.lammps import preload_libmpi
preload_libmpi()
from pyfield.optimizers import run_optimizer
from pyfield.diagnostics import cost_breakdown


def _print_breakdown(report):
    print(f'total cost = {report.total_cost:.4f}')
    print('(per-target table:)')
    for r in report.target_reports:
        sim_id = r.description.split()[0].lstrip('+')[2:]
        target = r.target if r.target is not None else 0
        print(f'  {sim_id:<22}  FF={r.value:>9.4f}  '
              f'target={target:>9.4f}  residual={r.residual:>9.2f}')


print('=' * 70)
print(f'BEFORE {populated.optimizer.method.upper()} — initial Chenoweth-2008 ReaxFF vs B3LYP/def2-SVP targets')
print('=' * 70)
before = cost_breakdown(populated)
_print_breakdown(before)

result = run_optimizer(populated)

print()
print('=' * 70)
print(f'AFTER {populated.optimizer.method.upper()}  — best FF written to {result.best_ffield_path}')
print('=' * 70)
after = cost_breakdown(populated, ffield_path=result.best_ffield_path)
_print_breakdown(after)
print()
print(f'final cost: {result.final_cost:.4f}')
print(f'cost trace length: {len(result.cost_trace)}')
print(f'reduction: {before.total_cost:.1f} → {after.total_cost:.1f}  '
      f'({100*(1-after.total_cost/before.total_cost):.1f}% drop)')

## 7. Cost trace

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(result.cost_trace, marker='.', linewidth=0.6)
ax.set_xlabel('SA step (best across walkers)')
ax.set_ylabel('cost')
ax.set_title('Water ReaxFF refit — SA cost trace')
ax.set_yscale('log')
fig.tight_layout()
fig.savefig('examples/_water_cost_trace.png', dpi=80)

## 8. Reproducibility

Every step is content-keyed (constraint included in the QM cache hash), and SA proposes deterministically from the master RNG, so re-running the entire pipeline gives bit-identical results.

In [ ]:
cfg2 = load_yaml('tests/water_train.yaml')
relaxed2, jr2 = relax_structures(cfg2)
scanned2, _ = expand_scans(relaxed2)
populated2, jp2 = populate_qm(scanned2)
result2 = run_optimizer(populated2)

assert all(hit for _, hit, _ in jr2), 'qm-relax should be all cache hits'
assert all(hit or 'reuse_relax_energy' in a for a, hit, _ in jp2), \
    'qm-prep should be all cache hits or reuse-relax-energy'
for t1, t2 in zip(populated.targets, populated2.targets):
    assert t1.__pydantic_extra__['target'] == t2.__pydantic_extra__['target']
assert result.final_cost == result2.final_cost, (result.final_cost, result2.final_cost)

print(f'qm-relax re-run: {sum(1 for _, h, _ in jr2 if h)}/{len(jr2)} cache hits')
print(f'qm-prep  re-run: {sum(1 for _, h, _ in jp2 if h)}/{len(jp2)} cache hits + reuse_relax')
print(f'{populated.optimizer.method.upper()} re-run cost: {result2.final_cost}  (matches first run)')
print('OK — qm-relax + make-scan + qm-prep + parallel optimiser chain is bit-reproducible.')